# Chapitre 7 : Création d'applications de chat
## Démarrage rapide avec l'API OpenAI

Ce carnet est adapté du [dépôt d'exemples Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) qui inclut des carnets accédant aux services [Azure OpenAI](notebook-azure-openai.ipynb).

L'API Python OpenAI fonctionne également avec les modèles Azure OpenAI, avec quelques modifications. En savoir plus sur les différences ici : [Comment basculer entre les points de terminaison OpenAI et Azure OpenAI avec Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Vue d'ensemble  
"Les grands modèles de langage sont des fonctions qui associent du texte à du texte. Étant donné une chaîne de texte en entrée, un grand modèle de langage tente de prédire le texte qui suivra"(1). Ce carnet « démarrage rapide » présentera aux utilisateurs les concepts de haut niveau des LLM, les exigences essentielles des packages pour débuter avec AML, une introduction légère à la conception de prompts, ainsi que plusieurs courts exemples de divers cas d’utilisation. 


## Table des matières  

[Présentation](#overview)  
[Comment utiliser le service OpenAI](#how-to-use-openai-service)  
[1. Créer votre service OpenAI](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Identifiants](#3.-credentials)  

[Cas d'utilisation](#use-cases)    
[1. Résumer un texte](#1.-summarize-text)  
[2. Classifier un texte](#2.-classify-text)  
[3. Générer de nouveaux noms de produits](#3.-generate-new-product-names)  
[4. Affiner un classificateur](#4.fine-tune-a-classifier)  

[Références](#references)


### Construisez votre premier prompt  
Ce court exercice vous fournira une introduction de base pour soumettre des prompts à un modèle OpenAI pour une tâche simple de « résumé ».


**Étapes** :  
1. Installez la bibliothèque OpenAI dans votre environnement python  
2. Chargez les bibliothèques d’assistance standard et configurez vos identifiants de sécurité OpenAI habituels pour le service OpenAI que vous avez créé  
3. Choisissez un modèle pour votre tâche  
4. Créez un prompt simple pour le modèle  
5. Soumettez votre requête à l’API du modèle !


### 1. Installer OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importer les bibliothèques d'aide et instancier les identifiants


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Trouver le bon modèle  
Des modèles comme GPT-4o et GPT-4o mini peuvent comprendre et générer un langage naturel, et sont disponibles sur la plateforme OpenAI avec différents niveaux de puissance et de vitesse adaptés à différentes tâches.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Conception des invites  

"La magie des grands modèles de langage réside dans le fait qu’en étant entraînés à minimiser cette erreur de prédiction sur d’immenses quantités de texte, les modèles finissent par apprendre des concepts utiles pour ces prédictions. Par exemple, ils apprennent des concepts tels que"(1) :

* comment épeler
* comment fonctionne la grammaire
* comment paraphraser
* comment répondre aux questions
* comment tenir une conversation
* comment écrire en plusieurs langues
* comment coder
* etc.

#### Comment contrôler un grand modèle de langage  
"Parmi toutes les entrées d’un grand modèle de langage, de loin la plus influente est l’invite textuelle(1).

Les grands modèles de langage peuvent être sollicités pour produire une sortie de plusieurs manières :

Instruction : Dites au modèle ce que vous voulez
Complétion : Amenez le modèle à compléter le début de ce que vous voulez
Démonstration : Montrez au modèle ce que vous voulez, avec soit :
Quelques exemples dans l’invite
Des centaines ou milliers d’exemples dans un jeu de données d’entraînement pour l’ajustement fin"



#### Il existe trois règles de base pour créer des invites :

**Montrez et dites**. Soyez clair sur ce que vous voulez, soit par des instructions, des exemples, ou une combinaison des deux. Si vous voulez que le modèle classe une liste d’éléments par ordre alphabétique ou qu’il classe un paragraphe par sentiment, montrez-lui que c’est ce que vous souhaitez.

**Fournissez des données de qualité**. Si vous essayez de construire un classificateur ou de faire suivre un modèle d’un schéma, assurez-vous qu’il y ait suffisamment d’exemples. Veillez à relire vos exemples — le modèle est habituellement assez intelligent pour détecter les fautes d’orthographe basiques et vous donner une réponse, mais il peut aussi supposer que c’est intentionnel et cela peut affecter la réponse.

**Vérifiez vos paramètres.** Les réglages temperature et top_p contrôlent à quel point le modèle est déterministe dans la génération d’une réponse. Si vous lui demandez une réponse où il n’y a qu’une bonne réponse, vous voudrez les régler plus bas. Si vous cherchez des réponses plus variées, vous voudrez peut-être les régler plus haut. L’erreur numéro un que font les gens avec ces réglages est de supposer qu’ils sont des contrôles de “malice” ou de “créativité”.


Source : https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Soumettre !


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Répétez le même appel, comment les résultats se comparent-ils ?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Résumer le texte  
#### Défi  
Résumez un texte en ajoutant un « tl;dr : » à la fin d’un passage de texte. Observez comment le modèle comprend comment effectuer plusieurs tâches sans instructions supplémentaires. Vous pouvez expérimenter avec des invites plus descriptives que « tl;dr » pour modifier le comportement du modèle et personnaliser le résumé que vous recevez(3).  

Des travaux récents ont démontré des gains substantiels sur de nombreuses tâches et benchmarks de TAL en pré-entraînement sur un grand corpus de texte, suivis d’un ajustement (fine-tuning) sur une tâche spécifique. Bien que typiquement agnostique à la tâche dans l’architecture, cette méthode nécessite toujours des jeux de données d’ajustement spécifiques à la tâche comprenant des milliers voire des dizaines de milliers d’exemples. En revanche, les humains peuvent généralement accomplir une nouvelle tâche linguistique à partir de seulement quelques exemples ou d’instructions simples – ce que les systèmes de TAL actuels ont encore largement du mal à faire. Nous montrons ici que l’augmentation de la taille des modèles de langue améliore grandement la performance agnostique à la tâche en few-shot, atteignant parfois même une compétitivité avec les approches de fine-tuning de pointe antérieures. 



Tl;dr


# Exercices pour plusieurs cas d'utilisation  
1. Résumer un texte  
2. Classer un texte  
3. Générer de nouveaux noms de produits


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Classifier du texte  
#### Défi  
Classer des éléments dans des catégories fournies au moment de l'inférence. Dans l'exemple suivant, nous fournissons à la fois les catégories et le texte à classer dans l'invite(*playground_reference). 

Demande client : Bonjour, une des touches de mon clavier d'ordinateur portable s'est récemment cassée et je vais avoir besoin d'un remplacement : 

Catégorie classée :


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Générer de Nouveaux Noms de Produits
#### Défi
Créez des noms de produits à partir de mots exemples. Ici, nous incluons dans l’invite des informations sur le produit pour lequel nous allons générer des noms. Nous fournissons également un exemple similaire pour montrer le modèle que nous souhaitons recevoir. Nous avons aussi réglé la valeur de température élevée pour augmenter l'aléatoire et obtenir des réponses plus innovantes.

Description du produit : Un mixeur à milkshake pour la maison
Mots de départ : rapide, sain, compact.
Noms de produits : HomeShaker, Fit Shaker, QuickShake, Shake Maker

Description du produit : Une paire de chaussures pouvant s'adapter à toute taille de pied.
Mots de départ : adaptable, ajusté, omni-ajusté.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Références  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Meilleures pratiques pour l’affinage des modèles GPT afin de classifier le texte](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pour plus d'aide  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Contributeurs
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
